In [206]:
!pip install neo4j
!pip install ultralytics

In [207]:
!rm -rf /content/Pre-registro

In [208]:
!git clone https://github.com/SinthiaG/Pre-registro

Cloning into 'Pre-registro'...
remote: Enumerating objects: 171, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 171 (delta 9), reused 22 (delta 8), pack-reused 141 (from 2)
Receiving objects: 100% (171/171), 77.35 MiB | 17.02 MiB/s, done.
Resolving deltas: 100% (109/109), done.
Updating files: 100% (117/117), done.


In [243]:
%%writefile /content/Pre-registro/.env

OPENAI_API_KEY=XXX
NEO4J_URI=neo4j+s://2e998866.databases.neo4j.io
NEO4J_USERNAME=2e998866
NEO4J_PASSWORD=XXXX
PATH_MODEL=models/car-parts/best_espanol.pt

Overwriting /content/Pre-registro/.env


In [210]:
from dotenv import load_dotenv
import os
from enum import Enum
from typing import Optional, List
from pydantic import BaseModel, Field
from datetime import date
import random
from pathlib import Path
from openai import OpenAI
import json
from neo4j import GraphDatabase # Import GraphDatabase

In [211]:
load_dotenv("/content/Pre-registro/.env")

True

In [212]:
uri = os.getenv("NEO4J_URI")
user = os.getenv("NEO4J_USERNAME")
print(uri)
print(user)

neo4j+s://2e998866.databases.neo4j.io
2e998866


In [213]:
# -----------------------
# Enums
# -----------------------
class EstadoDocumentoEnum(str, Enum):
    REGISTRADO = "Registrado"
    APROBADO = "Aprobado"
    CERRADO = "Cerrado"

class TipoVehiculoEnum(str, Enum):
    AUTO = "Auto"
    CAMION = "Camión"
    MOTOCICLETA = "Motocicleta"
    VAN = "Van"
    OTRO = "Otro"

# -----------------------
# Clases Pydantic
# -----------------------

class Vehiculo(BaseModel):
    """
    Representa un vehículo con su información identificativa y características.
    """
    placa: str = Field(..., description="Número de matrícula del vehículo.")
    vin: Optional[str] = Field(None, description="Número de Identificación del Vehículo (VIN).")
    marca: Optional[str] = Field(None, description="Marca del vehículo.")
    modelo: Optional[str] = Field(None, description="Modelo del vehículo.")
    anio: Optional[int] = Field(None, description="Año de fabricación del vehículo.")
    color: Optional[str] = Field(None, description="Color del vehículo.")
    tipo: Optional[TipoVehiculoEnum] = Field(None, description="Tipo de vehículo.")
    kilometraje_actual: Optional[int] = Field(None, description="Kilometraje actual del vehículo.")


class Propietario(BaseModel):
    """
    Representa un propietario de un vehículo.
    """
    nombre_completo: str = Field(..., description="Nombre completo del propietario.")
    cedula: Optional[str] = Field(None, description="Número de cédula o identificación nacional.")
    telefono: Optional[str] = Field(None, description="Número de teléfono del propietario.")
    direccion: Optional[str] = Field(None, description="Dirección física del propietario.")

class OrdenIngreso(BaseModel):
    """
    Representa el documento principal de ingreso de un vehículo.
    """
    orden_id: str = Field(..., description="Identificador único de la orden de ingreso.")
    fecha_ingreso: date = Field(..., description="Fecha en que el vehículo fue ingresado.")
    declaracion_veracidad: Optional[str] = Field(None, description="Declaración de veracidad proporcionada por el propietario.")
    observaciones: Optional[str] = Field(None, description="Observaciones adicionales o comentarios.")
    estado_documento: Optional[EstadoDocumentoEnum] = Field(None, description="Estado actual del documento.")
    fecha_registro: Optional[date] = Field(None, description="Fecha en que se registró el documento en el sistema.")
    vehiculo: Vehiculo = Field(..., description="Vehículo asociado a la orden de ingreso.")
    propietario: Propietario = Field(..., description="Propietario asociado a la orden de ingreso.")



class TipoAccidenteEnum(str, Enum):
    CHOQUE = "Choque"
    ATROPELLO = "Atropello"
    COLISION_LATERAL = "Colisión lateral"
    VOLCAMIENTO = "Volcamiento"
    ESTRELLAMIENTO = "Estrellamiento"
    OTRO = "Otro"

class NivelGravedadEnum(str, Enum):
    LEVE = "Leve"
    MODERADO = "Moderado"
    GRAVE = "Grave"

class DanosEnum(str, Enum):
    GUARDACHOQUE_DELANTERO = "Guardachoque delantero"
    GUARDACHOQUE_POSTERIOR = "Guardachoque posterior"
    CAPO = "Capó"
    PUERTAS = "Puertas"
    GUARDAFANGO = "Guardafango"
    LUCES_DELANTERAS = "Luces delanteras"
    LUCES_POSTERIORES = "Luces posteriores"
    PARABRISAS = "Parabrisas"
    NEUMATICOS = "Neumáticos"
    CHASIS = "Chasis"
    MOTOR = "Motor"
    OTRO = "Otro"

class Accidente(BaseModel):
    """
    Representa un accidente en el que se ve involucrado un vehículo.
    """
    fecha_accidente: date = Field(..., description="Fecha en que ocurrió el accidente.")
    descripcion: Optional[str] = Field(None, description="Descripción detallada del accidente.")
    lugar: Optional[str] = Field(None, description="Lugar donde ocurrió el accidente.")
    tipo_accidente: Optional[TipoAccidenteEnum] = Field(None, description="Tipo o categoría del accidente.")
    gravedad: Optional[NivelGravedadEnum] = Field(None, description="Gravedad del accidente.")
    danos_reportados: Optional[List[DanosEnum]] = Field(default_factory=list, description="Lista de daños reportados al vehículo.")


In [214]:
import pandas as pd
import logging
import json
import xml.etree.ElementTree as ET
from neo4j.exceptions import ServiceUnavailable
from neo4j import GraphDatabase # Import GraphDatabase


class ConectorNeo4J:

    def __init__(self, uri, auth):
        self.driver = GraphDatabase.driver(uri, auth=auth)
        self.driver.verify_connectivity()

    def close(self):
        self.driver.close()

    def __del__(self):
        if hasattr(self, 'driver') and self.driver:
            self.driver.close()

    # -------------------------
    # CONSULTAS SIN PARÁMETROS
    # -------------------------
    def run_query(self, query_string):
        with self.driver.session() as session:
            result = session.run(query_string)
            records = list(result)

            print(f"Size Found: {len(records)}")

            df = pd.DataFrame([dict(record) for record in records])
            return df

    # -------------------------
    # CONSULTAS CON PARÁMETROS
    # -------------------------
    def run_query_param(self, query_string, query_param):
        with self.driver.session() as session:
            result = session.run(query_string, query_param)
            records = list(result)

            print(f"Size Found: {len(records)}")

            df = pd.DataFrame([dict(record) for record in records])
            return df

    # -------------------------
    # EJECUCIÓN AVANZADA
    # -------------------------
    def execute_query_with_params(self, query_string, parameters):
        summary, _, records = self.driver.execute_query(
            query_string,
            parameters_=parameters
        )

        print(f"Nodes created: {summary.counters.nodes_created}")
        return records

    # -------------------------
    # INGESTA TARIFARIO JSON
    # -------------------------
    def crear_tarifario_neo4j(self, json_path, query_string, clear=True):

        # 1. Cargar JSON
        with open(json_path, 'r') as f:
            tar = json.load(f)

        # 2. Ejecutar en Neo4j con sesión (mejor práctica)
        with self.driver.session() as session:
            result = session.run(query_string, data=tar)
            summary = result.consume()

        # 3. Retornar métricas
        return summary
    # -------------------------
    # CARGA XML FACTURAS
    # -------------------------
    def cargar_facturas(self, factura, query_factura, query_detalle, order_id):
        with self.driver.session() as session:

            # Parse XML
            root = ET.fromstring(factura)

            fid = root.find('infoTributaria/FactId').text

            oid = ""
            marca = ""
            modelo = ""

            for c in root.findall('infoAdicional/campoAdicional'):
                if c.get('nombre') == 'OrderID':
                    oid = c.text
                if c.get('nombre') == 'MARCA':
                    marca = c.text
                if c.get('nombre') == 'MODELO':
                    modelo = c.text

            # -------------------------
            # FACTURA
            # -------------------------
            result_factura = session.run(
                query_factura,
                fid=fid,
                oid=oid,
                marca=marca,
                modelo=modelo
            )

            summary_factura = result_factura.consume()

            # -------------------------
            # DETALLES
            # -------------------------
            total_detalles = 0
            total_relaciones = 0

            for d in root.findall('detalles/detalle'):

                result_detalle = session.run(
                    query_detalle,
                    fid=fid,
                    desc=d.find('descripcion').text,
                    p=float(d.find('precioUnitario').text)
                )

                summary_detalle = result_detalle.consume()

                total_detalles += summary_detalle.counters.nodes_created
                total_relaciones += summary_detalle.counters.relationships_created

            # -------------------------
            # RESUMEN FINAL
            # -------------------------
            return {
                "factura_id": fid,
                "orden_id": oid,
                "marca": marca,
                "modelo": modelo,
                "nodos_factura": summary_factura.counters.nodes_created,
                "relaciones_factura": summary_factura.counters.relationships_created,
                "nodos_detalles": total_detalles,
                "relaciones_detalles": total_relaciones,
                "total_nodos": summary_factura.counters.nodes_created + total_detalles,
                "total_relaciones": summary_factura.counters.relationships_created + total_relaciones
            }

In [215]:
ruta = Path("/content/Pre-registro/data/OI-2026-1088")
print(ruta)
ID_ORDEN_INGRESO  = ruta.name
print(ID_ORDEN_INGRESO)

/content/Pre-registro/data/OI-2026-1088
OI-2026-1088


In [216]:
from pathlib import Path

def read_file_content(file_dir_path: Path) -> str:
    """
    Reads the content of the 'Orden.txt' file from the specified order directory path.
    """
    with open(file_dir_path, 'r') as file:
        return file.read()

In [217]:
contenido_orden = read_file_content("/content/Pre-registro/data/OI-2026-1088/Orden.txt")
contenido_orden

'ORDEN DE INGRESO DEL VEHÍCULO\nINFORMACIÓN GENERAL\nEl presente formulario tiene como finalidad registrar la recepción e ingreso del vehículo, así como recopilar información relevante relacionada con la identificación del automotor y propietario.\nToda la información proporcionada será utilizada exclusivamente para fines administrativos, técnicos, legales y de control interno.\n\nDATOS DE LA ORDEN DE INGRESO\nN° Orden de Ingreso: OI-2026-1088\nFecha de Ingreso: 22 de mayo de 2026\nEstado del Documento:\n☒ Registrado\n☐ Aprobado\n☐ Cerrado\nFecha de Registro del Documento: 22/05/2026 11:00 AM\nObservaciones Generales: El vehículo ingresa para mantenimiento preventivo de los 85,000 km y revisión del sistema de frenos por presentar ruidos al frenar. Carrocería en buen estado, incluye llanta de emergencia y herramientas básicas.\n\n\nDATOS DEL PROPIETARIO\nNombres y Apellidos: Carlos Alejandro Mendoza Ruiz\nCédula / Identificación: 1723456789\nTeléfono: 099999999\nDirección: Pifo\nCorreo 

In [218]:
system_message = """Eres un experto en extraer información estructurada a partir de formularios y reportes de órdenes de ingreso de vehículos. Identifica detalles clave como los datos de la orden, la información del propietario y las especificaciones técnicas del automóvil. Presenta la información extraída en un formato claro y estructurado. Sé conciso, enfocándote en los datos esenciales del registro del automotor e ignorando el texto instructivo o de relleno innecesario."""

In [219]:
# You need to set your OpenAI API key. Replace 'YOUR_API_KEY' with your actual key.
# Alternatively, ensure the OPENAI_API_KEY environment variable is set.
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [220]:
def extract(document,clase= None, model="gpt-4o-2024-08-06", temperature=0):
    response = client.beta.chat.completions.parse(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": document},
        ],
        response_format=clase,
    )
    return json.loads(response.choices[0].message.content)

In [221]:
orden_ingreso = extract(contenido_orden,OrdenIngreso)
orden_ingreso['orden_id'] = ID_ORDEN_INGRESO
orden_ingreso

{'orden_id': 'OI-2026-1088',
 'fecha_ingreso': '2205-05-22',
 'declaracion_veracidad': None,
 'observaciones': 'El vehículo ingresa para mantenimiento preventivo de los 85,000 km y revisión del sistema de frenos por presentar ruidos al frenar. Carrocería en buen estado, incluye llanta de emergencia y herramientas básicas.',
 'estado_documento': 'Registrado',
 'fecha_registro': '2205-05-22',
 'vehiculo': {'placa': 'ZSY-7187',
  'vin': '93HFX84A7GH123456',
  'marca': 'Toyota',
  'modelo': 'RAV4',
  'anio': 2019,
  'color': 'Gris Plata',
  'tipo': 'Otro',
  'kilometraje_actual': 85420},
 'propietario': {'nombre_completo': 'Carlos Alejandro Mendoza Ruiz',
  'cedula': '1723456789',
  'telefono': '099999999',
  'direccion': 'Pifo'}}

In [222]:
import_orden_ingreso = """
WITH $data AS order_data

// ======================================================
// ORDEN DE INGRESO
// ======================================================

MERGE (orden:OrdenIngreso {ordenId: order_data.orden_id})

SET orden += {
    fechaIngreso: order_data.fecha_ingreso,
    declaracionVeracidad: order_data.declaracion_veracidad,
    observaciones: order_data.observaciones,
    estadoDocumento: order_data.estado_documento,
    fechaRegistro: order_data.fecha_registro
}

WITH orden, order_data

// ======================================================
// VEHÍCULO
// ======================================================

MERGE (vehiculo:Vehiculo {
    vehiculoId: randomUUID()
})

SET vehiculo += {
    placa: order_data.vehiculo.placa,
    vin: order_data.vehiculo.vin,
    marca: order_data.vehiculo.marca,
    modelo: order_data.vehiculo.modelo,
    anio: order_data.vehiculo.anio,
    color: order_data.vehiculo.color,
    tipo: order_data.vehiculo.tipo,
    kilometrajeActual: order_data.vehiculo.kilometraje_actual
}

WITH orden, vehiculo, order_data

// ======================================================
// PROPIETARIO
// ======================================================

MERGE (propietario:Propietario {
    propietarioId: randomUUID()
})

SET propietario += {
    cedula: order_data.propietario.cedula,
    nombreCompleto: order_data.propietario.nombre_completo,
    telefono: order_data.propietario.telefono,
    direccion: order_data.propietario.direccion
}

// ======================================================
// RELACIONES
// ======================================================

// Relación Propietario -> Vehículo
MERGE (propietario)-[:POSEE]->(vehiculo)

// Relación Vehículo -> OrdenIngreso
MERGE (vehiculo)-[:INGRESA]->(orden)

RETURN
    propietario,
    vehiculo,
    orden
"""

In [223]:
AUTH = (os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))
app = ConectorNeo4J(os.getenv("NEO4J_URI"), AUTH)

In [224]:
# Call the execute_query method directly from the driver instance,
# passing the 'data' parameter as a keyword argument as expected by the Cypher query.
records_orden_ingreso = app.driver.execute_query(
    query_=import_orden_ingreso,
    data=orden_ingreso
)
records_orden_ingreso

EagerResult(records=[<Record propietario=<Node element_id='4:4a9cea96-e718-4066-9789-91e03f975711:147' labels=frozenset({'Propietario'}) properties={'propietarioId': '3d94d7b7-ad65-421e-a778-a2383f6f3ce5', 'cedula': '1723456789', 'direccion': 'Pifo', 'nombreCompleto': 'Carlos Alejandro Mendoza Ruiz', 'telefono': '099999999'}> vehiculo=<Node element_id='4:4a9cea96-e718-4066-9789-91e03f975711:146' labels=frozenset({'Vehiculo'}) properties={'marca': 'Toyota', 'tipo': 'Otro', 'color': 'Gris Plata', 'vehiculoId': 'd0dfa9a2-d2e4-4a03-8835-a6ae75e828f5', 'kilometrajeActual': 85420, 'vin': '93HFX84A7GH123456', 'modelo': 'RAV4', 'anio': 2019, 'placa': 'ZSY-7187'}> orden=<Node element_id='4:4a9cea96-e718-4066-9789-91e03f975711:145' labels=frozenset({'OrdenIngreso'}) properties={'estadoDocumento': 'Registrado', 'fechaIngreso': '2205-05-22', 'fechaRegistro': '2205-05-22', 'observaciones': 'El vehículo ingresa para mantenimiento preventivo de los 85,000 km y revisión del sistema de frenos por prese

In [225]:
contenido_declaracion = read_file_content("/content/Pre-registro/data/OI-2026-1088/Declaracion.txt")
contenido_declaracion

'INFORMACIÓN DE ACCIDENTES DECLARADOS\n\nFecha del Accidente: 22 de mayo de 2026\n\nLugar del Accidente: Avenida Simón Bolívar y Ruta Viva, Quito\n\nTipo de Accidente:\n☒ Choque\n☐ Atropello\n☐ Colisión lateral\n☐ Volcamiento\n☐ Estrellamiento\n☐ Otro: _______________________\n\nNivel de Gravedad:\n☐ Leve\n☒ Moderado\n☐ Grave\n\nDescripción del Accidente: El vehículo frenó intempestivamente debido al tráfico de la zona y fue impactado en la parte posterior por otro automotor que no alcanzó a detenerse, desplazándolo hacia adelante y haciéndolo rozar lateralmente contra la baranda de seguridad.\n\nDaños Reportados en el Vehículo:\n☐ Guardachoque delantero\n☒ Guardachoque posterior\n☐ Capó\n☒ Puertas\n☒ Guardafango\n☐ Luces delanteras\n☒ Luces posteriores\n☐ Parabrisas\n☐ Neumáticos\n☐ Chasis\n☐ Motor\n☐ Otro: _______________________\n\nDetalle de los daños observados: El guardachoque posterior se encuentra desprendido y roto en su lado izquierdo; la luz posterior izquierda está totalmen

In [226]:
accidente_declarado = extract(contenido_declaracion,Accidente)
accidente_declarado

{'fecha_accidente': '2205-05-22',
 'descripcion': 'El vehículo frenó intempestivamente debido al tráfico de la zona y fue impactado en la parte posterior por otro automotor que no alcanzó a detenerse, desplazándolo hacia adelante y haciéndolo rozar lateralmente contra la baranda de seguridad.',
 'lugar': 'Avenida Simón Bolívar y Ruta Viva, Quito',
 'tipo_accidente': 'Choque',
 'gravedad': 'Moderado',
 'danos_reportados': ['Guardachoque posterior',
  'Puertas',
  'Guardafango',
  'Luces posteriores']}

In [227]:
import_accidente = """WITH $data AS accident_data

// ======================================================
// BUSCAR ORDEN Y VEHÍCULO
// ======================================================

MATCH (vehiculo:Vehiculo)-[:INGRESA]->(orden:OrdenIngreso)
WHERE orden.ordenId = $orden_id

// ======================================================
// ACCIDENTE
// ======================================================

MERGE (accidente:Accidente {
    accidenteId: randomUUID()
})

SET accidente += {
    fechaAccidente: accident_data.fecha_accidente,
    descripcion: accident_data.descripcion,
    lugar: accident_data.lugar,
    tipoAccidente: accident_data.tipo_accidente,
    gravedad: accident_data.gravedad,
    fechaRegistro: datetime()
}

// Relación orden -> accidente
MERGE (orden)-[:REGISTRA]->(accidente)

// Relación vehículo -> accidente
MERGE (vehiculo)-[:INVOLUCRADO_EN]->(accidente)

WITH vehiculo, orden, accidente, accident_data

// ======================================================
// DAÑOS REPORTADOS
// ======================================================

UNWIND coalesce(accident_data.danos_reportados, []) AS dano_nombre

MERGE (dano:Dano {
    danoId: randomUUID(),
    nombre: trim(toUpper(dano_nombre))
})

MERGE (accidente)-[:REPORTA]->(dano)

WITH vehiculo, orden, accidente, collect(dano) AS danos

RETURN {
    vehiculo: vehiculo,
    orden: orden,
    accidente: accidente,
    danos: danos
} AS resultado"""

In [228]:
# Call the execute_query method directly from the driver instance,
# passing the 'data' parameter as a keyword argument as expected by the Cypher query.
records_declaracion = app.driver.execute_query(
    query_=import_accidente,
    data = accidente_declarado,
    orden_id = ID_ORDEN_INGRESO
)
records_declaracion

EagerResult(records=[<Record resultado={'orden': <Node element_id='4:4a9cea96-e718-4066-9789-91e03f975711:145' labels=frozenset({'OrdenIngreso'}) properties={'estadoDocumento': 'Registrado', 'fechaIngreso': '2205-05-22', 'fechaRegistro': '2205-05-22', 'observaciones': 'El vehículo ingresa para mantenimiento preventivo de los 85,000 km y revisión del sistema de frenos por presentar ruidos al frenar. Carrocería en buen estado, incluye llanta de emergencia y herramientas básicas.', 'ordenId': 'OI-2026-1088'}>, 'accidente': <Node element_id='4:4a9cea96-e718-4066-9789-91e03f975711:148' labels=frozenset({'Accidente'}) properties={'descripcion': 'El vehículo frenó intempestivamente debido al tráfico de la zona y fue impactado en la parte posterior por otro automotor que no alcanzó a detenerse, desplazándolo hacia adelante y haciéndolo rozar lateralmente contra la baranda de seguridad.', 'gravedad': 'Moderado', 'fechaAccidente': '2205-05-22', 'fechaRegistro': neo4j.time.DateTime(2026, 5, 25, 2

In [229]:
print(orden_ingreso['orden_id'],ID_ORDEN_INGRESO)

OI-2026-1088 OI-2026-1088


###a.2 Generación de Tarifario y Facturas Parametrizadas
Asociamos las facturas generadas al `ID_ORDEN_INGRESO` extraído

### a.3 Creación Scripts insercion base de datos neo4j

In [230]:
query_tarifario = """UNWIND $data AS row

MERGE (ma:Marca {nombre: row.marca})
MERGE (mo:Modelo {nombre: row.modelo, anio: row.anio})
MERGE (s:Siniestro {tipo: row.tipo_siniestro})
MERGE (i:Item {nombre: row.item})

MERGE (ma)-[:TIENE_MODELO]->(mo)
MERGE (mo)-[:SUFRE]->(s)

MERGE (s)-[r:INCLUYE]->(i)
SET r.precio = row.precio_acordado,
    r.categoria = row.categoria"""

In [231]:
tarifario_summary = app.crear_tarifario_neo4j("/content/Pre-registro/bd_aseguradora/tarifario_seguros.json",
                                              query_tarifario)
tarifario_summary

In [232]:
query_factura = """
               MERGE (f:Factura {facId: $fid})
                MERGE (o:OrdenIngreso {ordenId: $oid})
                MERGE (f)-[:BASADA_EN]->(o)

                WITH f, $marca AS marca, $modelo AS modelo

                MATCH (mo:Modelo {nombre: modelo})
                MATCH (ma:Marca {nombre: marca})

                MERGE (ma)-[:TIENE_MODELO]->(mo)
                MERGE (f)-[:PARA_MODELO]->(mo)
            """

In [233]:
query_detalle = """MATCH (f:Factura {facId: $fid})
                    CREATE (df:DetalleFactura {desc: $desc, p: $p})
                    CREATE (f)-[:CONTIENE]->(df)"""


In [234]:
factura =  read_file_content("/content/Pre-registro/data/OI-2026-1088/Factura.xml")
factura


'<?xml version="1.0" ?>\n<factura id="comprobante" version="2.1.0">\n  <infoTributaria>\n    <ambiente>2</ambiente>\n    <tipoEmision>1</tipoEmision>\n    <razonSocial>TALLER LOS PINOS S.A.</razonSocial>\n    <nombreComercial>TALLER LOS PINOS S.A.</nombreComercial>\n    <ruc>1792145632001</ruc>\n    <FactId>FAC088</FactId>\n    <codDoc>01</codDoc>\n    <estab>001</estab>\n    <ptoEmi>010</ptoEmi>\n    <secuencial>111315088</secuencial>\n    <dirMatriz>AV. DE LOS GRANADOS</dirMatriz>\n  </infoTributaria>\n  <infoFactura>\n    <fechaEmision>25/05/2026</fechaEmision>\n    <dirEstablecimiento>AV. DE LOS GRANADOS</dirEstablecimiento>\n    <obligadoContabilidad>SI</obligadoContabilidad>\n    <tipoIdentificacionComprador>05</tipoIdentificacionComprador>\n    <razonSocialComprador>ASEGURADORA GENIAL S.A.</razonSocialComprador>\n    <identificacionComprador>1790010020001</identificacionComprador>\n    <direccionComprador>AV. AMAZONAS N32</direccionComprador>\n    <totalSinImpuestos>3685.50</tot

In [235]:
resultado = app.cargar_facturas(factura=factura, query_factura=query_factura, query_detalle=query_detalle, order_id=ID_ORDEN_INGRESO)
print(f"Factura ingestion summary: {resultado}")
resultado

Factura ingestion summary: {'factura_id': 'FAC088', 'orden_id': 'OI-2026-1088', 'marca': 'Toyota', 'modelo': 'RAV4', 'nodos_factura': 1, 'relaciones_factura': 2, 'nodos_detalles': 6, 'relaciones_detalles': 6, 'total_nodos': 7, 'total_relaciones': 8}


{'factura_id': 'FAC088',
 'orden_id': 'OI-2026-1088',
 'marca': 'Toyota',
 'modelo': 'RAV4',
 'nodos_factura': 1,
 'relaciones_factura': 2,
 'nodos_detalles': 6,
 'relaciones_detalles': 6,
 'total_nodos': 7,
 'total_relaciones': 8}

In [236]:
import zipfile
import os

zip_file_path = os.getenv("NEO4J_USERNAME")

In [237]:
import os
import json
import cv2
from ultralytics import YOLO

def analizar_imagen(ruta_imagen, ruta_modelo):
    modelo = YOLO(ruta_modelo)
    resultados = modelo.predict(source=ruta_imagen, save=False, conf=0.25)
    resultado = resultados[0]
    nombre_base = os.path.splitext(os.path.basename(ruta_imagen))[0]
    reporte_json = {
        "total_detecciones": len(resultado.boxes),
        "detecciones": []
    }

    print("\n=========================================")
    print("      REPORTE DE DAÑOS DETECTADOS        ")
    print("=========================================")

    if len(resultado.boxes) == 0:
        print("No se detectó ningún daño en el vehículo.")
    else:
        for box in resultado.boxes:
            id_clase = int(box.cls)
            nombre_dano = modelo.names[id_clase]
            confianza = float(box.conf) * 100
            print(f"-> {nombre_dano.upper()} (Confianza: {confianza:.2f}%)")
            reporte_json["detecciones"].append({
                "descripcion": nombre_dano,
                "porcentaje_confianza": round(confianza, 2)
            })

    print("=========================================\n")
    return reporte_json

In [238]:
import pandas as pd
import os

def procesar_evidencias(imagenes_dir, ruta_modelo, app_connector, orden_ingreso_id):
    all_image_reports = []

    if not ruta_modelo:
        print("Error: PATH_MODEL environment variable not set or ruta_modelo is empty.")
        return pd.DataFrame(), []

    print(f"Iniciando análisis de imágenes en: {imagenes_dir}")
    for filename in os.listdir(imagenes_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
            image_path = os.path.join(imagenes_dir, filename)
            print(f"Analizando imagen: {image_path}")
            reporte = analizar_imagen(image_path, ruta_modelo)
            reporte["nombre_imagen"] = filename # Add image filename to the report
            all_image_reports.append(reporte)
    return all_image_reports




In [239]:
# Call the function with your specific parameters
imagenes_dir = f"/content/Pre-registro/data/{ID_ORDEN_INGRESO}/imagenes"
ruta_modelo = "/content/Pre-registro/models/car-parts/best_espanol.pt"

# Assuming 'app' and 'ID_ORDEN_INGRESO' are defined in previous cells

data_image_reports = procesar_evidencias(imagenes_dir, ruta_modelo, app, ID_ORDEN_INGRESO)
data_image_reports

Iniciando análisis de imágenes en: /content/Pre-registro/data/OI-2026-1088/imagenes
Analizando imagen: /content/Pre-registro/data/OI-2026-1088/imagenes/imagen1.jpg

image 1/1 /content/Pre-registro/data/OI-2026-1088/imagenes/imagen1.jpg: 512x640 1 abolladura-capo, 1 abolladura-parachoques-del, 5076.2ms
Speed: 12.7ms preprocess, 5076.2ms inference, 1.7ms postprocess per image at shape (1, 3, 512, 640)

      REPORTE DE DAÑOS DETECTADOS        
-> ABOLLADURA-CAPO (Confianza: 87.26%)
-> ABOLLADURA-PARACHOQUES-DEL (Confianza: 60.48%)

Analizando imagen: /content/Pre-registro/data/OI-2026-1088/imagenes/imagen2.jpg

image 1/1 /content/Pre-registro/data/OI-2026-1088/imagenes/imagen2.jpg: 448x640 2 abolladura-puerta-exteriors, 2336.8ms
Speed: 3.1ms preprocess, 2336.8ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 640)

      REPORTE DE DAÑOS DETECTADOS        
-> ABOLLADURA-PUERTA-EXTERIOR (Confianza: 64.01%)
-> ABOLLADURA-PUERTA-EXTERIOR (Confianza: 33.22%)



[{'total_detecciones': 2,
  'detecciones': [{'descripcion': 'abolladura-capo',
    'porcentaje_confianza': 87.26},
   {'descripcion': 'abolladura-parachoques-del',
    'porcentaje_confianza': 60.48}],
  'nombre_imagen': 'imagen1.jpg'},
 {'total_detecciones': 2,
  'detecciones': [{'descripcion': 'abolladura-puerta-exterior',
    'porcentaje_confianza': 64.01},
   {'descripcion': 'abolladura-puerta-exterior',
    'porcentaje_confianza': 33.22}],
  'nombre_imagen': 'imagen2.jpg'}]

In [240]:
import_imagen ="""WITH $data AS imagenes

UNWIND imagenes AS imagen_data

CREATE (img:Imagen {
    imagenId: randomUUID(),
    nombre: imagen_data.nombre_imagen,
    totalDetecciones: imagen_data.total_detecciones
})

WITH img, imagen_data

MATCH (vehiculo:Vehiculo)-[:INGRESA]->(orden:OrdenIngreso)
MATCH (vehiculo)-[:INVOLUCRADO_EN]->(accidente:Accidente)
WHERE orden.ordenId = $orden_id

MERGE (accidente)-[:TIENE_IMAGEN]->(img)

WITH img, imagen_data

UNWIND imagen_data.detecciones AS det

WITH img, det

CREATE (d:Deteccion {
    deteccionId: randomUUID(),
    descripcion: det.descripcion,
    confianza: det.porcentaje_confianza
})

CREATE (img)-[:TIENE_DETECCION]->(d)
"""

In [241]:
records_imagenes = app.driver.execute_query(
    query_=import_imagen,
    data = data_image_reports,
    orden_id = ID_ORDEN_INGRESO
)
records_imagenes

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x7b6f1e93aba0>, keys=[])